In [ ]:
import pandas as pd
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification, AutoTokenizer, set_seed
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint
!pip install iterative-stratification
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [ ]:
drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Learning_Curve_epochs"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


Loading the dataset (full size)

In [ ]:
with zipfile.ZipFile("aapd.zip") as z:
    with z.open("aapd.json") as f:
        aapd = json.load(f)

In [ ]:
aapd_df_train = pd.DataFrame(aapd["data"]["train"])
aapd_df_val = pd.DataFrame(aapd["data"]["val"])
aapd_df_test = pd.DataFrame(aapd["data"]["test"])

In [ ]:
mlb = joblib.load("mlb.joblib")

In [ ]:
#reusing the aapd's mlb
aapd_y_train = mlb.transform(aapd_df_train["labels"])
aapd_y_val   = mlb.transform(aapd_df_val["labels"])
aapd_y_test  = mlb.transform(aapd_df_test["labels"])

In [ ]:
aapd_y_train.shape, aapd_y_val.shape, aapd_y_test.shape #ok

((53840, 54), (1000, 54), (1000, 54))

In [ ]:
aapd_X_train = aapd_df_train["text"]
aapd_X_val   = aapd_df_val["text"]
aapd_X_test  = aapd_df_test["text"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# for DistilBERT max token length is 512 - the longest abstract has 522 words, so truncation will happen
def tokenize(texts):
    return tokenizer(texts.tolist(), padding="max_length", truncation=True, max_length=512)

train_enc = tokenize(aapd_X_train)
dev_enc   = tokenize(aapd_X_val)
test_enc  = tokenize(aapd_X_test)

In [ ]:
y_train_bin = aapd_y_train.astype(np.float32)
y_dev_bin   = aapd_y_val.astype(np.float32)
y_test_bin  = aapd_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)

['Adaptation and Self-Organizing Systems' 'Applications'
 'Artificial Intelligence' 'Combinatorics' 'Computation and Language'
 'Computational Complexity'
 'Computational Engineering, Finance, and Science'
 'Computational Geometry' 'Computational Linguistics'
 'Computer Science and Game Theory'
 'Computer Vision and Pattern Recognition' 'Computers and Society'
 'Cryptography and Security' 'Data Analysis, Statistics and Probability'
 'Data Structures and Algorithms' 'Databases' 'Digital Libraries'
 'Discrete Mathematics' 'Disordered Systems and Neural Networks'
 'Distributed, Parallel, and Cluster Computing'
 'Formal Languages and Automata Theory' 'Human-Computer Interaction'
 'Information Retrieval' 'Information Theory (Computer Science)'
 'Information Theory (Mathematics)' 'Logic' 'Logic in Computer Science'
 'Machine Learning (Computer Science)' 'Machine Learning (Statistics)'
 'Mathematical Software' 'Methodology' 'Multiagent Systems' 'Multimedia'
 'Networking and Internet Architect

In [ ]:
##loading the nested subsets created for FFT DistilBERT learning curve
#subset_indices_path = os.path.join("/content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Learning_Curve/aapd_learning_curve_indices.npz")
#loaded_subsets = np.load(subset_indices_path)
#subsets = {int(key): loaded_subsets[key].astype(int) for key in loaded_subsets.files}

In [ ]:
#Learning curve nested subset selection
#multilabel stratification as in  Sechidis et al (2011)
def select_subset(parent_indices, labels, target_size, random_state=42):
    parent_indices = np.asarray(parent_indices)
    parent_labels = labels[parent_indices]

    splitter = MultilabelStratifiedShuffleSplit(n_splits=1, train_size=target_size,test_size = len(parent_indices) - target_size, random_state=random_state)

    dummy_X = np.zeros((len(parent_indices), 1))

    selected_positions, _ = next(splitter.split(dummy_X, parent_labels))

    # enforcing the exact requested size of the subset - this is a small tradeoff between exact subset size and optimal multilabel stratification
    #but differences were tiny when the size was not enforced;
    if len(selected_positions) < target_size:
        remaining_positions = np.setdiff1d(
            np.arange(len(parent_indices)),
            selected_positions)

        rng = np.random.default_rng(random_state)

        additional_positions = rng.choice(
            remaining_positions,
            size=target_size - len(selected_positions),
            replace=False)

        selected_positions = np.concatenate(
            [selected_positions, additional_positions])

    elif len(selected_positions) > target_size:
        rng = np.random.default_rng(random_state)

        selected_positions = rng.choice(
            selected_positions,
            size=target_size,
            replace=False )

    return parent_indices[selected_positions]

In [ ]:
#saving one fixed set of nested subsets for each training seed/experiment
#every smaller training subset is fully contained within every larger one

subset_indices_path = os.path.join(output_dir,"aapd_learning_curve_indices.npz")

if os.path.exists(subset_indices_path):
    loaded = np.load(subset_indices_path)

    subsets = {int(size): loaded[size] for size in loaded.files}

    print("Loaded existing nested subsets.")

else:
    all_indices = np.arange(len(y_train_bin))
    subsets = {len(all_indices): all_indices.copy()}
    parent_indices = all_indices.copy()

    for target_size in [10000, 5000, 2500, 1000, 500]:
        parent_indices = select_subset(
            parent_indices=parent_indices,
            labels=y_train_bin,
            target_size=target_size,
            random_state=42)
        subsets[target_size] = parent_indices.copy()

    np.savez(
        subset_indices_path,
        **{
            str(size): indices
            for size, indices in subsets.items()})

    print("Created and saved nested subsets.")

Loaded existing nested subsets.


In [ ]:
training_sizes = [500, 1000, 2500, 5000, 10000]

for size in training_sizes:
    print(size, len(subsets[size]))

for smaller, larger in zip(training_sizes[:-1], training_sizes[1:]):
    nested = set(subsets[smaller]).issubset(set(subsets[larger]))
    print(f"{smaller} nested in {larger}: {nested}")
#subsets are correctly nested

500 500
1000 1000
2500 2500
5000 5000
10000 10000
500 nested in 1000: True
1000 nested in 2500: True
2500 nested in 5000: True
5000 nested in 10000: True


In [ ]:
#label distributions
full_prevalence = y_train_bin.mean(axis=0)

coverage_rows = []

for size in training_sizes:
    subset_labels = y_train_bin[subsets[size]]
    subset_prevalence = subset_labels.mean(axis=0)

    coverage_rows.append({
        "training_size": size,
        "average_labels_per_document":
            subset_labels.sum(axis=1).mean(),
        "labels_with_zero_examples":
            int((subset_labels.sum(axis=0) == 0).sum()),
        "mean_absolute_prevalence_difference":
            np.abs(subset_prevalence - full_prevalence).mean()
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(os.path.join(output_dir, "aapd_learning_curve_coverage.csv"), index=False)
coverage_df

,training_size,average_labels_per_document,labels_with_zero_examples,mean_absolute_prevalence_difference
0,500,2.4100,0,0.001022
1,1000,2.4000,0,0.000543
2,2500,2.4028,0,0.000340
3,5000,2.3936,0,0.000301
4,10000,2.4078,0,0.000041


In [ ]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int)

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro}

In [ ]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, training_dataset, training_size, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)


    if measure_vram:
        reset_cuda_peak_memory()


    model = AutoModelForSequenceClassification.from_pretrained(
        config["base_model"],
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id)
    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none")

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=training_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"])])


#for resuming if something goes wrong
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


#If training resumes from a checkpoint (in case of disconnected runtime), this records only the resumed portion, not the time spent before interruption.
#Final timing comparisons use uninterrupted runs only.
    sync_cuda()
    train_start = time.perf_counter()
#includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)

    sync_cuda()
    train_time_sec = time.perf_counter() - train_start
    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "AAPD",
        "method": "full_finetuning",
        "training_size": training_size,
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        # single forward pass — gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        # derive predictions - needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(config["output_dir"], f"classification_report_{training_size}_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(
                config["output_dir"],
                f"test_predictions_{training_size}_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

In [ ]:
#fixed params
learning_curve_config = {
    "base_model": "distilbert-base-uncased",
    "tokenizer_name": "distilbert-base-uncased",

    "max_length": 512,
    "num_train_epochs": 30, #higher than for full size
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 5, #higher than for full size

    "learning_rate": 3e-05, #best params from the search, used in FFT
    "batch_size": 8}

**Train and test**

In [ ]:
training_seeds = [0, 1, 2]

learning_curve_output_dir = os.path.join(output_dir, "aapd_learning_curve_fft")
os.makedirs(learning_curve_output_dir, exist_ok=True)

results_path = os.path.join(learning_curve_output_dir, "AAPD_DistilBERT_FFT_learning_curve_results.csv")

if os.path.exists(results_path):
    existing_results = pd.read_csv(results_path)

    existing_results = (existing_results.drop_duplicates(subset=["training_size", "seed"],
            keep="last").sort_values(["training_size", "seed"]).reset_index(drop=True))

    learning_curve_results = existing_results.to_dict("records")

else:
    existing_results = pd.DataFrame()
    learning_curve_results = []

for training_size in training_sizes:
    subset_indices = subsets[training_size].tolist()
    training_subset = train_dataset.select(subset_indices)

    for seed in training_seeds:


        if not existing_results.empty:
            already_done = existing_results[(existing_results["training_size"] == training_size) & (existing_results["seed"] == seed)]

            if not already_done.empty:
                print(f"Skipping size={training_size}, seed={seed}")
                continue

        config = learning_curve_config.copy()
        config["output_dir"] = os.path.join(
            learning_curve_output_dir,
            f"size_{training_size}",
            f"seed_{seed}")

        result = run_training(
            config=config,
            training_dataset=training_subset,
            training_size=training_size,
            seed=seed,
            evaluate_test=True,
            measure_vram=True,
            save_report=True)

        learning_curve_results.append(result)

        pd.DataFrame(learning_curve_results).to_csv(results_path, index=False)

Skipping size=500, seed=0
Skipping size=500, seed=1
Skipping size=500, seed=2
Skipping size=1000, seed=0
Skipping size=1000, seed=1
Skipping size=1000, seed=2
Skipping size=2500, seed=0
Skipping size=2500, seed=1
Skipping size=2500, seed=2
Skipping size=5000, seed=0
Skipping size=5000, seed=1
Skipping size=5000, seed=2
Skipping size=10000, seed=0
Skipping size=10000, seed=1


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
0.002798,0.107192,20,0.716515,0.535966


Classification report saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Learning_Curve_epochs/aapd_learning_curve_fft/size_10000/seed_2/classification_report_10000_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/AAPD_DistilBERT_Learning_Curve_epochs/aapd_learning_curve_fft/size_10000/seed_2/test_predictions_10000_seed_2.npz


In [ ]:
results_df = pd.read_csv(results_path)

learning_curve_summary = (
    results_df.groupby("training_size").agg(
        macro_f1_mean=("test_f1_macro", "mean"),
        macro_f1_std=("test_f1_macro", "std"),
        micro_f1_mean=("test_f1_micro", "mean"),
        micro_f1_std=("test_f1_micro", "std"),
        training_time_mean=("train_time_sec", "mean"),
        training_time_std=("train_time_sec", "std"),
        epochs_mean=("actual_epochs_trained", "mean")).reset_index())

learning_curve_summary.to_csv(os.path.join(output_dir, "aapd_learning_curve_summary.csv"), index=False)
learning_curve_summary



,training_size,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,training_time_mean,training_time_std,epochs_mean
0,500,0.161395,0.009775,0.503521,0.007108,1645.953053,109.671430,30.000000
1,1000,0.304221,0.007815,0.590774,0.007018,1982.476476,103.470534,29.333333
2,2500,0.423957,0.007028,0.649675,0.004040,2533.854641,320.086276,28.000000
3,5000,0.470775,0.010243,0.670210,0.007069,2988.430568,749.227282,25.666667
4,10000,0.520721,0.004509,0.688788,0.006036,3729.375449,1129.815720,22.333333


In [ ]:
from google.colab import runtime
runtime.unassign()